In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!rm -rf Cap3_MedicionPobreza
!git clone https://github.com/LizamaD/Cap3_MedicionPobreza.git

Cloning into 'Cap3_MedicionPobreza'...
remote: Enumerating objects: 142, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 142 (delta 74), reused 119 (delta 51), pack-reused 0 (from 0)
Receiving objects: 100% (142/142), 4.39 MiB | 60.70 MiB/s, done.
Resolving deltas: 100% (74/74), done.


In [3]:
import pandas as pd

path_bases = "/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/"
path_pobreza = '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Base final/pobreza_24.csv'


# --- Carga tus datos crudos ---
poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")
trabajos_raw = pd.read_csv(f"{path_bases}trabajos.csv")
gastospersona_raw = pd.read_csv(f"{path_bases}gastospersona.csv")
gastoshogar_raw = pd.read_csv(f"{path_bases}gastoshogar.csv")
ingresos_raw = pd.read_csv(f"{path_bases}ingresos.csv")
pobreza_raw = pd.read_csv(path_pobreza)

/tmp/ipykernel_32418/1323284465.py:8: DtypeWarning: Columns (45) have mixed types. Specify dtype option on import or set low_memory=False.
  poblacion_raw = pd.read_csv(f"{path_bases}poblacion.csv")
/tmp/ipykernel_32418/1323284465.py:9: DtypeWarning: Columns (1,4,27,44) have mixed types. Specify dtype option on import or set low_memory=False.
  viviendas_raw = pd.read_csv(f"{path_bases}viviendas.csv")
/tmp/ipykernel_32418/1323284465.py:10: DtypeWarning: Columns (45,49,53,61,63,69,75,77,81,83,85) have mixed types. Specify dtype option on import or set low_memory=False.
  hogares_raw = pd.read_csv(f"{path_bases}hogares.csv")


In [4]:
%cd /content/Cap3_MedicionPobreza

/content/Cap3_MedicionPobreza


In [5]:
from src.trabajos import process_trabajos
from src.ingresos import generar_ingreso_deflactado_ago2024
from src.gastoshogar import procesar_gastos_enigh
from src.gastospersona import procesar_gastos_persona_enigh
from src.hogares import process_hogares
from src.poblacion import process_poblacion
from src.viviendas import process_viviendas
from src.pipeline import create_master_table, impute_data

In [6]:
# --- Crea la tabla maestra ---
# Esta función procesa, une todo y guarda el resultado en "master_table.csv"
master_df = create_master_table(
    pob_keys=pobreza_raw,
    pob_df=poblacion_raw,
    viv_df=viviendas_raw,
    hog_df=hogares_raw,
    trab_df=trabajos_raw,
    gasper_df=gastospersona_raw,
    gashog_df=gastoshogar_raw,
    ing_df=ingresos_raw,
    output_path=f"{path_bases}enigh_master_table.csv" # Ruta sugerida
)

# --- Imputa los datos faltantes ---
# Esta función toma la tabla maestra y la deja lista para el modelado
df_final_limpio = impute_data(master_df)

# --- Verifica el resultado ---
print("\nDataFrame final después de la imputación:")
print(df_final_limpio.head())

# Confirma que no hay valores nulos
print("\nSuma de nulos por columna en el DF final:")
print(df_final_limpio.isnull().sum().sum()) # Debería ser 0


Procesando tablas individuales...


/content/Cap3_MedicionPobreza/src/ingresos.py:34: FutureWarning: The provided callable <function sum at 0x79cf23730360> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  aguinaldo = pd.pivot_table(


Uniendo tablas en una tabla maestra...
Tabla maestra creada con 308444 filas y 160 columnas.
Guardando tabla maestra en '/content/drive/MyDrive/Doctorado_DavidLizama/datos_tesis/coneval/2024/Bases de datos/enigh_master_table.csv'...
Iniciando proceso de imputación de datos...
  - Categórica 'socios': Imputando NaNs con '__MISSING__'.
  - Categórica 'soc_nr1': Imputando NaNs con '__MISSING__'.
  - Categórica 'soc_nr2': Imputando NaNs con '__MISSING__'.
  - Categórica 'soc_resp': Imputando NaNs con '__MISSING__'.
  - Categórica 'otra_act': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact2': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact3': Imputando NaNs con '__MISSING__'.
  - Categórica 'tipoact4': Imputando NaNs con '__MISSING__'.
  - Categórica 'lugar': Imputando NaNs con '__MISSING__'.
  - Categórica 'conf_pers': Imputando NaNs con '__MISSING__'.
  - Categórica 'inst': Imputando NaNs con '__MISSING__'.
  - Numérica 'ing_mon': Imputando NaNs con mediana (17668.34) 

/content/Cap3_MedicionPobreza/src/pipeline.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean[flag_col_name] = df_clean[col].isnull().astype(int)
/content/Cap3_MedicionPobreza/src/pipeline.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_clean[flag_col_name] = df_clean[col].isnull().astype(int)
/content/Cap3_MedicionPobreza/src/pipeline.py:112: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all

0


In [7]:
master_df.head(2)

,folioviv,foliohog,numren,pobreza,pobreza_e,parentesco,sexo,edad,afrod,hablaind,...,conf_pers,gasto_tri_y,gas_nm_tri_y,gasto_educ_total,gasto_salud_ind,inscrip,colegia,inst,gasto_persona_total,tiene_gasto_educ
0,100001901,1,1,0.0,0.0,101,1,32,0,0,...,,0.0,20571.4,0.0,0.0,0.0,0.0,nan,20571.4,0.0
1,100001901,1,2,0.0,0.0,201,2,24,0,0,...,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df_final_limpio.head(2)

,folioviv,foliohog,numren,pobreza,pobreza_e,parentesco,sexo,edad,afrod,hablaind,...,no_ing_imputed,tiene_suel_imputed,gasto_tri_y_imputed,gas_nm_tri_y_imputed,gasto_educ_total_imputed,gasto_salud_ind_imputed,inscrip_imputed,colegia_imputed,gasto_persona_total_imputed,tiene_gasto_educ_imputed
0,100001901,1,1,0.0,0.0,101,1,32,0,0,...,1,1,0,0,0,0,0,0,0,0
1,100001901,1,2,0.0,0.0,201,2,24,0,0,...,1,1,1,1,1,1,1,1,1,1


In [39]:
df_final_limpio.iloc[:,80:90].head(2)

,num_table,autocons,regalos,remunera,transferen,indice_conectividad,score_iaas,gasto_tri_x,gas_nm_tri_x,imujer_tri
0,0,2,1,1,2,2,1,39988.34,21214.25,1797.15
1,0,2,1,1,2,2,1,39988.34,21214.25,1797.15


In [51]:
df_final_limpio.gasto_tri_x.value_counts()

,count
gasto_tri_x,
0.00,59
102871.47,20
69275.50,20
46339.28,18
54712.80,17
...,...
11059.39,1
8302.15,1
9544.25,1


In [12]:
columnas_texto = df_final_limpio.select_dtypes(include=['object', 'string', 'category']).columns
print(columnas_texto)

Index(['parentesco', 'tipo_viv', 'tenencia', 'socios', 'soc_nr1', 'soc_nr2',
       'soc_resp', 'otra_act', 'tipoact2', 'tipoact3', 'tipoact4', 'lugar',
       'conf_pers', 'inst'],
      dtype='object')
